# 14.8 · Transformer 时序预测 / Transformers for Time Series

> **课程定位 / Where this fits**
> 第 8 课，**Part 14 · 时间序列**。把 Part 12 的 Transformer 用到预测。
> Lesson 8, **Part 14 · Time Series**. Bringing Part 12's Transformer to forecasting.
>
> Transformer(12.3)的自注意力能**一步直连任意两个时间点**, 天生适合捕捉长依赖, 是近年时序预测的热门方向。但直接套用有两个难题: ①注意力是 $O(n^2)$, 长序列**太贵**;②逐点当 token 会**丢失局部语义**且序列太长。**PatchTST** 给出关键解法——**把序列切成 patch(小段)**, 每个 patch 当一个 token(像 ViT 切图像块, 10.9), 既缩短序列、又保留局部模式。本课讲清这些思想, 并搭一个 **patch-based Transformer** 预测器。
> The Transformer's (12.3) self-attention **directly connects any two time points in one hop**, ideal for long dependencies and a hot area in TS forecasting. But naive use has two issues: ① attention is $O(n^2)$, **too costly** for long series; ② treating each point as a token **loses local semantics** and makes sequences too long. **PatchTST** gives the key fix — **split the series into patches**, each patch a token (like ViT's image patches, 10.9) — shortening the sequence while preserving local patterns. We explain these ideas and build a **patch-based Transformer** forecaster.
>
> 💼 **实战/面试视角**："注意力用于时序的优势与O(n²)问题 / patching(PatchTST)解决什么 / Informer的稀疏注意力 / Transformer vs RNN for TS" 是前沿时序常考。
> 💼 **Practical/interview angle:** "attention for TS / the O(n²) problem / what patching solves / Informer's sparse attention / Transformer vs RNN for TS" — frontier questions.

> 📐 **符号约定 / Notation**
> - patch —— 序列切出的连续小段, 当作一个 token / a contiguous segment treated as a token
> - $O(n^2)$ —— 自注意力对序列长度的复杂度 / attention's cost in sequence length

> 💡 **面试相关 / Interview-relevant**
> - "自注意力用于时序的优点(任意距离)+缺点(O(n²))"（出镜率 ★★★★）
> - "PatchTST 的 patching 解决什么"（★★★★★）
> - "Informer 的稀疏注意力/为什么"（★★★）
> - "Transformer vs RNN/TCN 用于时序"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解自注意力用于时序的优势与 O(n²) 难题。
   Understand attention's strengths for TS and the O(n²) challenge.
2. 掌握 **patching(PatchTST)** 思想及其好处。
   Master the patching (PatchTST) idea and its benefits.
3. **搭一个 patch-based Transformer** 预测器。
   Build a patch-based Transformer forecaster.
4. 了解 Informer 等长序列 Transformer 与适用场景。
   Know Informer-style long-sequence Transformers and use cases.

## 目录 / TOC
1. [注意力用于时序 + O(n²)难题 ⭐](#1)
2. [Patching:把序列切成块 ⭐](#2)
3. [搭 patch-Transformer 预测 ⭐](#3)
4. [Informer 等 + 何时用 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 注意力用于时序 + O(n²)难题 ⭐ / Attention for TS & the O(n²) Challenge

回忆 12.3: 自注意力让序列里**任意两个位置一步直接交互**。用在时序上, 这意味着预测时能**直接关注很久以前的相关时刻**(如去年同月), 不像 RNN 要逐步传递、也不像 TCN 受感受野限制。这是 Transformer 做时序的最大吸引力。
Recall 12.3: self-attention lets **any two positions interact directly in one hop**. For TS, this means a forecast can **directly attend to relevant moments long ago** (last year's same month), unlike RNN's step relay or TCN's receptive-field limit. This is the big appeal.

但有代价(面试核心)：
But there's a cost (interview core):
- **$O(n^2)$ 复杂度**:注意力要算**每对位置**的相关性。序列长度 $n$ 翻倍, 计算和显存涨 4 倍。对**长序列**(几千上万步, 如高频/长历史数据)非常昂贵。
  **$O(n^2)$ cost:** attention scores **every pair** of positions. Doubling length $n$ quadruples compute/memory. Very expensive for **long sequences** (thousands of steps).
- **逐点 token 的问题**:若把**每个时间点**当一个 token, ①序列就和原始一样长(O(n²)更糟);②单个数值点**几乎没有语义**(不像一个词)。
  **Point-token problem:** treating **each time point** as a token makes ① the sequence as long as the raw data (worse O(n²)); ② a single value carries **almost no semantics** (unlike a word).

所以时序 Transformer 的研究核心就是**怎么降低这个成本、增强语义**。最有效的思路之一是 **patching**。
So TS-Transformer research centers on **reducing this cost and enriching semantics**. One of the most effective ideas is **patching**.


<a id="2"></a>
## 2. Patching:把序列切成块 ⭐ / Patching: Split into Segments

**PatchTST**(2023)的关键洞察(借鉴 ViT, 10.9): 不要把**单个时间点**当 token, 而是把序列切成**一段段连续的小块(patch)**, **每个 patch 当一个 token**。比如长度 96 的序列, 用 patch 长 16、步幅 8, 就变成约 11 个 token。
**PatchTST**'s (2023) key insight (borrowing from ViT, 10.9): don't make a **single point** a token; split the series into **contiguous segments (patches)**, **each patch a token**. E.g. a length-96 series with patch length 16, stride 8 → about 11 tokens.

好处(面试)：
Benefits (interview):
- **序列大幅变短**:token 数从 $n$ 降到 $n/\text{stride}$ → 注意力的 $O(n^2)$ 成本大降, 能处理更长历史。
  **Much shorter sequence:** tokens drop from $n$ to $n/\text{stride}$ → far lower $O(n^2)$ cost, longer history possible.
- **保留局部语义**:一个 patch(一小段子序列)比单个点**更有信息量**(含局部趋势/形状), 像"词"而非"字母"。
  **Local semantics:** a patch (a sub-series) is **more informative** than a single point (local trend/shape), like a "word" not a "letter."
- 注意力则在 patch 之间建模**长距离**依赖。局部由 patch 抓, 全局由注意力抓——分工明确。
  Attention then models **long-range** dependencies across patches. Local by patches, global by attention.

下面可视化把序列切成 patch。
Let's visualize patching the series.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, math, time
import torch, torch.nn as nn, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid"); torch.manual_seed(0)
ap=[112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432]
ts=np.array(ap,dtype=float)

# 可视化 patching: 长度48的窗口切成 patch / visualize patching a length-48 window
L, patch_len, stride = 48, 8, 8
seg = ts[:L]
n_patches = (L - patch_len)//stride + 1
fig, ax = plt.subplots(figsize=(11, 3.6)); ax.plot(seg, color="gray", alpha=0.4)
colors = plt.cm.tab10(np.linspace(0,1,n_patches))
for i in range(n_patches):
    s = i*stride
    ax.plot(range(s, s+patch_len), seg[s:s+patch_len], "o-", color=colors[i])
ax.set_title(f"Patching: 长度{L}的序列 → {n_patches}个patch(每段{patch_len}点); 每个patch当一个token")
plt.tight_layout(); plt.show()
print(f"序列长度{L} → {n_patches}个patch-token (而非{L}个point-token)")
print("好处: ①token数大减→注意力O(n²)成本降 ②每个patch含局部形状(更有语义, 像'词')")
print("局部模式由patch捕捉, 长距离依赖由patch间的注意力捕捉 → 分工明确(PatchTST思想)")


<a id="3"></a>
## 3. 搭 patch-Transformer 预测 ⭐ / Build a Patch-Transformer Forecaster

把 PatchTST 的核心搭出来: **滑窗 → 切 patch → 每个 patch 线性投影成 token → Transformer 编码器 → 预测下一步**。复用 12.3 的 Transformer 组件。
Build PatchTST's core: **window → patch → linear-project each patch to a token → Transformer encoder → predict next step**. Reusing 12.3's components.


In [ ]:
idx = pd.date_range("1949-01", periods=len(ts), freq="MS")
train, test = ts[:120], ts[120:]
mn, mx = train.min(), train.max(); scale=lambda x:(x-mn)/(mx-mn); unscale=lambda x:x*(mx-mn)+mn
L, P = 24, 4                                              # 窗口24, patch长4 → 6个patch-token / window/patch
s_train = scale(train)
X, Y = [], []
for i in range(len(s_train)-L): X.append(s_train[i:i+L]); Y.append(s_train[i+L])
X = torch.tensor(np.array(X), dtype=torch.float32); Y = torch.tensor(np.array(Y), dtype=torch.float32).unsqueeze(-1)

class PatchTransformer(nn.Module):
    def __init__(self, L, P, d=32, heads=4):
        super().__init__()
        self.P = P; self.n_patch = L//P
        self.embed = nn.Linear(P, d)                      # 每个patch(P个点)线性投影成d维token / patch → token
        self.pos = nn.Parameter(torch.randn(1, self.n_patch, d)*0.02)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d); self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.head = nn.Linear(self.n_patch*d, 1)
    def forward(self, x):                                 # x: (batch, L)
        B = x.shape[0]
        patches = x.view(B, self.n_patch, self.P)         # 切成patch / split into patches
        h = self.embed(patches) + self.pos                # patch→token + 位置编码 / patch embedding + position
        a,_ = self.attn(self.norm1(h), self.norm1(h), self.norm1(h)); h = h + a   # 自注意力(残差) / self-attn
        h = h + self.ff(self.norm2(h))                    # FFN(残差) / FFN
        return self.head(h.reshape(B, -1))                # 展平所有patch → 预测下一步 / flatten → predict

torch.manual_seed(0); net = PatchTransformer(L, P); opt = torch.optim.Adam(net.parameters(), 3e-3); t0=time.time()
for ep in range(400): opt.zero_grad(); loss = F.mse_loss(net(X), Y); loss.backward(); opt.step()
net.eval(); window = list(s_train[-L:]); preds=[]
with torch.no_grad():
    for _ in range(len(test)):
        x = torch.tensor(window[-L:], dtype=torch.float32).view(1, L); p = net(x).item(); preds.append(p); window.append(p)
fc = unscale(np.array(preds)); mape = np.mean(np.abs((test-fc)/test))*100
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(idx[:120], train, label="训练"); ax.plot(idx[120:], test, label="真实", color="green")
ax.plot(idx[120:], fc, "r--", label="Patch-Transformer预测")
ax.legend(); ax.set_title(f"Patch-Transformer 递归预测: MAPE={mape:.1f}% (训练{time.time()-t0:.0f}s)")
plt.tight_layout(); plt.show()
print(f"Patch-Transformer 预测 MAPE = {mape:.1f}%")
print("结构: 滑窗→切patch→每patch投影成token→自注意力(patch间长依赖)→预测; 这是PatchTST的核心")


<a id="4"></a>
## 4. Informer 等 + 何时用 + 小结 ⭐ / Informer & When to Use

时序 Transformer 的研究主要在**解决 $O(n^2)$ 和长序列**(面试可举)：
TS-Transformer research mainly tackles **$O(n^2)$ and long sequences:**
- **Informer**(2021): 用**稀疏注意力(ProbSparse)** 只算最重要的一部分注意力对, 把 $O(n^2)$ 降到 $O(n\log n)$, 能做**长序列预测**。
  **Informer (2021):** **ProbSparse attention** computes only the most important pairs, cutting $O(n^2)$ to $O(n\log n)$ for **long-sequence forecasting**.
- **PatchTST**(2023, 本课实现的思想): patching + 通道独立, 简单却很强, 是近年强基线。
  **PatchTST (2023, what we built):** patching + channel-independence; simple yet strong, a recent top baseline.
- **Autoformer / FEDformer**: 把分解(14.2)和自相关/频域思想融入注意力。
  **Autoformer / FEDformer:** fold decomposition (14.2) and autocorrelation/frequency ideas into attention.

**何时用 Transformer 做时序(诚实)**:它们在**长序列、多变量、大数据**上有优势; 但**小数据上**(如本课 144 点)往往**打不过经典方法甚至简单 DLinear**——事实上有研究指出"一个简单线性模型在很多基准上能媲美复杂时序 Transformer", 提醒我们**别盲目上复杂模型**。先试经典/简单方法做基线, 数据量和复杂度够了再上 Transformer。
**When to use TS Transformers (honest):** they shine on **long sequences, multivariate, big data**; but on **small data** (144 points here) they often **lose to classical methods or even a simple linear model** — research famously showed "a simple linear model rivals complex TS Transformers on many benchmarks," a reminder **not to over-engineer**. Start with classical/simple baselines; reach for Transformers when data/complexity warrant.

```
注意力用于时序: 任意两时刻一步直连(长依赖好) 但 O(n²)对长序列贵 + 逐点token丢语义
PatchTST: 把序列切patch, 每patch当token(像ViT切图) → token数减少(降O(n²)) + 保留局部语义
结构: 滑窗→切patch→线性投影成token+位置编码→自注意力(patch间长依赖)→预测
Informer: 稀疏注意力ProbSparse 把O(n²)→O(nlogn), 做长序列; Autoformer/FEDformer融合分解/频域
诚实: 小数据上Transformer常打不过经典法甚至简单线性模型(DLinear); 别盲目上复杂模型
何时用: 长序列/多变量/大数据; 先简单基线, 不够再上Transformer
```

### 💡 面试速查 / Interview cheat-sheet
1. **注意力做时序**: 任意距离直连(长依赖好); 缺点O(n²)+逐点token丢语义。
   Attention for TS: any-distance links (good long-range); cons O(n²) + point-tokens lack semantics.
2. **PatchTST**: 切patch当token, 减token数(降成本)+保留局部语义。
   PatchTST: patches as tokens, fewer tokens (cheaper) + local semantics.
3. **Informer**: 稀疏注意力O(n²)→O(nlogn), 处理长序列。
   Informer: sparse attention O(n²)→O(nlogn) for long sequences.
4. **诚实**: 小数据上常输给经典法/简单线性模型(DLinear); 别过度工程。
   Honest: on small data often loses to classical/simple linear (DLinear); don't over-engineer.
5. **何时用**: 长序列/多变量/大数据才值; 先简单基线。
   When: long/multivariate/big data; baseline first.

### 下一节 / Next
**14.9 多变量 & 多步预测**——前面都是单变量。现实常是**多个相互影响的序列**(GDP/消费/投资)。我们会用 **VAR(向量自回归)** 同时建模多个序列, 并讲清**多步预测**的两种策略: 递归 vs 直接。
**14.9 Multivariate & Multi-step** — so far univariate. Reality often has **multiple interacting series** (GDP/consumption/investment). We'll use **VAR (vector autoregression)** to model several series jointly, and clarify **multi-step** strategies: recursive vs direct.
